# انحدار الغابة العشوائية — Google Colab

**الهدف:** التنبؤ بمتغير مستهدف متصل باستخدام **انحدار الغابة العشوائية** — **مجموعة** من أشجار القرار تقلّل الإفراط في التلائم وتحسّن الدقة.

| المثال | المتغير(ات) المستقل(ة) (X) | المتغير المستهدف (y) | مجموعة البيانات |
|---------|----------------|------------|---------|
| **المثال 1** | Level | Salary | `../Datasets/Position_Salaries.csv` |
| **المثال 2** | Area_sqft, Bedrooms, Age_years | Price | `../Datasets/house_price.csv` |

| المرحلة | الموضوع | الخلايا |
|-------|-------|-------|
| المرحلة 0 | الإعداد | التثبيت والاستيراد |
| — | دليل الخوارزمية | التجميع، العيّنة الذاتية، العشوائية في الميزات |
| المرحلة 1 | معالجة البيانات مسبقاً | تحميل → تنظيف → ترميز → تقسيم |
| المرحلة 2 | الخوارزمية | تدريب → تنبؤ → تصوير → تقييم |

> **التشغيل:** Runtime → Run all (أو Ctrl+F9)


---
# دليل الخوارزمية — انحدار الغابة العشوائية

## ما هي الغابة العشوائية؟

**الغابة العشوائية** = **عدة أشجار قرار** تُدرَّب على **مجموعات فرعية عشوائية مختلفة** من البيانات والميزات، ثم يُؤخذ **متوسط** تنبؤاتها.

إنها طريقة **تجميع Bagging** (Bootstrap Aggregating) اخترعها Leo Breiman (2001).

## كيف تعمل (خطوة بخطوة)

| الخطوة | الاسم | ماذا يحدث |
|------|------|--------------|
| 1 | **عيّنة Bootstrap** | سحب n صف **مع الاستبدال** من بيانات التدريب |
| 2 | **ميزات عشوائية** | عند كل تقسيم، يُنظر فقط إلى **مجموعة فرعية عشوائية** من الميزات |
| 3 | **نمو الشجرة** | بناء شجرة CART كاملة على تلك العيّنة الذاتية |
| 4 | **التكرار** | إنشاء `n_estimators` شجرة (مثلاً 100) |
| 5 | **التجميع** | التنبؤ النهائي = **متوسط** تنبؤات جميع الأشجار |

## قاعدة التنبؤ (الانحدار)

لعيّنة جديدة **x**:

`ŷ = (1/B) · Σ ŷ_b(x)`

حيث **B** = عدد الأشجار (`n_estimators`) و **ŷ_b** = تنبؤ الشجرة **b**.

## المعاملات الفائقة الرئيسية

| المعامل | الدور | التأثير |
|-----------|------|--------|
| `n_estimators` | عدد الأشجار في الغابة | المزيد من الأشجار → استقرار أكبر (عائد متناقص) |
| `max_depth` | أقصى عمق لكل شجرة | يحدّ من الإفراط في التلائم لكل شجرة |
| `min_samples_split` | أقل عدد عيّنات لتقسيم عقدة | أعلى → أشجار أبسط |
| `min_samples_leaf` | أقل عدد عيّنات في ورقة | أعلى → تنبؤات أنعم |
| `max_features` | الميزات المُراعاة عند كل تقسيم | `'sqrt'` أو `'1.0'` — يضيف تنوعاً بين الأشجار |
| `bootstrap` | استخدام عيّنة Bootstrap | `True` (افتراضي) — جوهر التجميع |
| `random_state` | البذرة العشوائية | غابة قابلة للتكرار |

## الغابة العشوائية مقابل شجرة قرار واحدة (CART)

| | CART واحدة | الغابة العشوائية |
|---|-------------|---------------|
| الأشجار | شجرة **1** | **عدة** أشجار (مجموعة) |
| الإفراط في التلائم | خطر **مرتفع** إذا كانت عميقة | **أقل** — المتوسط يقلّل التباين |
| التنبؤ | دالة خطوات | **متوسط أنعم** للخطوات |
| قابلية التفسير | سهلة (`plot_tree`) | أصعب — استخدم **أهمية الميزات** |
| التحجيم | غير مطلوب | **غير مطلوب** |
| السرعة | سريعة | أبطأ (تدريب B شجرة) |

## أهمية الميزات

تحسب الغابة العشوائية الأهمية بقياس مقدار تقليل كل ميزة **لـ MSE** عبر جميع التقسيمات في جميع الأشجار. قيمة أعلى = تأثير أكبر على التنبؤات.

## ما يجب أن يتذكره الطالب

1. الغابة العشوائية = **Bagging + مجموعات فرعية عشوائية للميزات** عند كل تقسيم.
2. التنبؤ النهائي = **متوسط** مخرجات جميع الأشجار (انحدار).
3. **لا حاجة لتحجيم الميزات** — كما في أشجار القرار.
4. المزيد من الأشجار (`n_estimators`) عادةً يساعد حتى تستقر الأداء.
5. استخدم **أهمية الميزات** لفهم أي المدخلات الأكثر أهمية.


## المرحلة 0 — الخلية 0: تثبيت المكتبات

يتضمن Google Colab عادةً معظم المكتبات. تضمن هذه الخلية توفر الحزم المطلوبة.

**ما تفعله هذه الخلية:** تثبيت scikit-learn و pandas و matplotlib و numpy و seaborn بصمت.


In [ ]:
# تثبيت المكتبات المطلوبة بصمت (-q يخفي المخرجات)
!pip install -q scikit-learn pandas matplotlib numpy seaborn


## المرحلة 0 — الخلية 1: استيراد المكتبات

استيراد المكتبات لمعالجة البيانات والتحضير المسبق والنمذجة والتقييم.

**ما تفعله هذه الخلية:** تحميل numpy و pandas و matplotlib و RandomForestRegressor من sklearn والمقاييس.


In [ ]:
# --- استيراد المكتبات ---
import numpy as np              # العمليات العددية والمصفوفات
import pandas as pd             # تحميل ومعالجة البيانات الجدولية
import matplotlib.pyplot as plt # إنشاء الرسوم البيانية
import seaborn as sns           # تصورات إحصائية (تنسيق اختياري)

from sklearn.model_selection import train_test_split       # تقسيم البيانات إلى تدريب/اختبار
from sklearn.impute import SimpleImputer                   # ملء القيم المفقودة
from sklearn.ensemble import RandomForestRegressor         # من regressor الغابة العشوائية التجميعي
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score  # مقاييس التقييم

plt.rcParams['figure.figsize'] = (10, 6)  # حجم الرسم الافتراضي: عرض=10، ارتفاع=6 بوصة
plt.rcParams['font.family'] = ['Segoe UI', 'Tahoma', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
sns.set_theme(style='whitegrid')            # خلفية بيضاء نظيفة مع خطوط شبكة
np.random.seed(42)                          # تثبيت البذرة العشوائية لتقسيمات قابلة للتكرار

print('المكتبات جاهزة')                      # تأكيد تحميل جميع الاستيرادات بنجاح


---
# المثال 1: رواتب المناصب — انحدار الغابة العشوائية

التنبؤ بـ **Salary** من **Level** الوظيفي. العلاقة **غير خطية** — الغابة العشوائية تُوسط عدة أشجار للحصول على تنبؤات أنعم من CART واحدة.

| العمود | الدور | الوصف |
|--------|------|-------------|
| `Level` | ميزة (X) | مستوى الوظيفة (1–10) |
| `Salary` | متغير مستهدف (y) | الراتب السنوي بالدولار |

**الملف:** `../Datasets/Position_Salaries.csv`


---
# المرحلة 1: معالجة البيانات مسبقاً

تحضير البيانات قبل التدريب — نفس القالب يُعاد استخدامه لخوارزميات أخرى.


## المثال 1 — الخلية 1: تحميل واستكشاف البيانات

تحميل ملف CSV وإجراء استكشاف أولي (head, info, describe, shape).

**ما تفعله هذه الخلية:** قراءة `../Datasets/Position_Salaries.csv` وعرض إحصائيات أساسية.


In [ ]:
# الخطوة 1) تحميل مجموعة البيانات
dataset = pd.read_csv('../Datasets/Position_Salaries.csv')  # قراءة CSV إلى DataFrame

FEATURE_COL = 'Level'   # المتغير المستقل (X) — مستوى الوظيفة
TARGET_COL = 'Salary'   # المتغير التابع (y) — الراتب المراد التنبؤ به

print('أول 5 صفوف:')          # طباعة تسمية للجدول أدناه
display(dataset.head())         # عرض أول 5 صفوف لفحص البيانات

print('\nمعلومات مجموعة البيانات:')       # طباعة تسمية لأنواع الأعمدة وعدد القيم الفارغة
dataset.info()                  # عرض أسماء الأعمدة وأنواع البيانات وعدد القيم غير الفارغة

print('\nملخص إحصائي:')  # طباعة تسمية للإحصائيات العددية
display(dataset.describe())       # عرض العدد والمتوسط والانحراف المعياري والحد الأدنى والأقصى والربيعيات

print(f'\nالشكل: {dataset.shape[0]} صف × {dataset.shape[1]} عمود')  # إجمالي الصفوف والأعمدة


## المثال 1 — الخلية 2: تنظيف البيانات (معالجة القيم المفقودة)

التحقق من القيم المفقودة، إزالة التكرارات، وتطبيق الإكمال إذا لزم الأمر.

**ما تفعله هذه الخلية:** تنظيف مجموعة البيانات قبل النمذجة.


In [ ]:
# الخطوة 2) تنظيف البيانات

print('القيم المفقودة لكل عمود:')  # طباعة تسمية
print(dataset.isnull().sum())        # عد NaN في كل عمود

rows_before = len(dataset)                              # تخزين عدد الصفوف قبل التنظيف
dataset = dataset.drop_duplicates().reset_index(drop=True)  # إزالة الصفوف المكررة
rows_after = len(dataset)                               # تخزين عدد الصفوف بعد إزالة التكرار
print(f'\nالتكرارات المُزالة: {rows_before - rows_after}')  # عرض عدد التكرارات

num_cols = dataset.select_dtypes(include=[np.number]).columns.tolist()  # الأعمدة العددية فقط
imputer = SimpleImputer(missing_values=np.nan, strategy='mean')  # ملء NaN بمتوسط العمود
if dataset.isnull().sum().sum() > 0:               # إذا وُجدت قيم مفقودة
    dataset[num_cols] = imputer.fit_transform(dataset[num_cols])  # إكمال الأعمدة العددية
    print('تم إكمال القيم المفقودة بالمتوسط')      # تأكيد الإكمال
else:
    print('لا توجد قيم مفقودة — لم يُطبَّق الإكمال')  # تخطّ عند اكتمال البيانات

print(f'\nالصفوف بعد التنظيف: {rows_after}')    # العدد النهائي للصفوف


## المثال 1 — الخلية 3: ترميز البيانات الفئوية

نستخدم `Level` (عددي). عمود `Position` نصي — يُتخطّى لأن Level يُرمّز المرتبة بالفعل.

**ما تفعله هذه الخلية:** التحقق من الأعمدة الفئوية وترميزها إذا لزم الأمر.


In [ ]:
# الخطوة 3) الترميز الفئوي

cat_cols = dataset.select_dtypes(include=['object', 'category']).columns.tolist()  # إيجاد الأعمدة النصية
print(f'الأعمدة الفئوية (غير مستخدمة كـ X): {cat_cols}')  # أسماء المناصب — للمعلومات فقط
print(f'الميزة المستخدمة للغابة العشوائية: {FEATURE_COL}')    # Level عددي — لا حاجة للترميز
print('لا يلزم ترميز — X عددي.')


## المثال 1 — الخلية 4: تقسيم البيانات

تعريف X (Level) و y (Salary)، ثم تقسيم 80/20.

**ما تفعله هذه الخلية:** إنشاء مصفوفات الميزات/المستهدف وتطبيق train_test_split.


In [ ]:
# الخطوة 4) تقسيم التدريب-الاختبار

X = dataset[[FEATURE_COL]].values  # مصفوفة الميزات: Level (مصفوفة ثنائية الأبعاد لـ sklearn)
y = dataset[TARGET_COL].values     # متجه المستهدف: قيم Salary

X_train, X_test, y_train, y_test = train_test_split(
    X, y,                  # البيانات المراد تقسيمها
    test_size=0.2,         # 20% اختبار، 80% تدريب
    random_state=42        # تقسيم قابل للتكرار
)

print(f'شكل X_train: {X_train.shape}')  # شكل ميزات التدريب
print(f'شكل X_test:  {X_test.shape}')   # شكل ميزات الاختبار
print(f'شكل y_train: {y_train.shape}')  # شكل أهداف التدريب
print(f'شكل y_test:  {y_test.shape}')   # شكل أهداف الاختبار


> **ملاحظة:** انحدار الغابة العشوائية **لا يتطلب** تحجيم الميزات. الأشجار تقسم وفق عتبات — المقياس لا يهم.


---
# المرحلة 2: انحدار الغابة العشوائية

تدريب غابة عشوائية للتنبؤ بـ Salary من Level.


## المثال 1 — الخلية 5: تدريب النموذج

تدريب `RandomForestRegressor` بـ `n_estimators=100` شجرة. كل شجرة ترى عيّنة bootstrap.

**ما تفعله هذه الخلية:** ملاءمة الغابة وطباعة عدد الأشجار.


In [ ]:
# الخطوة 5) تدريب Random Forest Regressor

regressor = RandomForestRegressor(
    n_estimators=100,      # عدد الأشجار في الغابة
    max_depth=4,           # تحديد عمق كل شجرة (مجموعة بيانات صغيرة)
    min_samples_leaf=1,    # أقل عدد عيّنات مطلوب في عقدة ورقة
    random_state=42,       # bootstrap وتقسيمات قابلة للتكرار
    n_jobs=-1              # استخدام جميع أنوية المعالج للتدريب الأسرع
)

regressor.fit(X_train, y_train)  # تدريب جميع الأشجار على عيّنات bootstrap

print('تم تدريب الغابة العشوائية بنجاح.')              # تأكيد اكتمال التدريب
print(f'عدد الأشجار (estimators): {len(regressor.estimators_)}')  # يجب أن يساوي n_estimators
print(f'أهمية الميزات: {regressor.feature_importances_}')        # الأهمية لكل ميزة


## المثال 1 — الخلية 6: التنبؤ

التنبؤ بـ Salary على مجموعة الاختبار بمتوسط تنبؤات جميع الأشجار.

**ما تفعله هذه الخلية:** إنشاء تنبؤات المجموعة.


In [ ]:
# الخطوة 6) التنبؤ

y_pred_train = regressor.predict(X_train)  # متوسط تنبؤات الأشجار لبيانات التدريب
y_pred_test = regressor.predict(X_test)    # متوسط تنبؤات الأشجار لبيانات الاختبار

print('عيّنة من التنبؤات (مجموعة الاختبار):')  # طباعة تسمية
for i in range(len(y_test)):             # عرض جميع تنبؤات الاختبار
    print(f'  Level={X_test[i][0]:.0f} -> فعلي=${y_test[i]:,.0f}، متوقع=${y_pred_test[i]:,.0f}')


## المثال 1 — الخلية 7: التصوير

رسم **منحنى تنبؤ الغابة العشوائية** (أنعم من CART واحدة) ومقارنته مع البيانات الخام.

**ما تفعله هذه الخلية:** عرض تنبؤات المجموعة مقابل مستويات الراتب الفعلية.


In [ ]:
# الخطوة 7) التصوير — منحنى RF + نقاط مبعثرة

fig, ax = plt.subplots(figsize=(10, 6))  # رسم واحد للوضوح

X_plot = np.linspace(X.min(), X.max(), 500).reshape(-1, 1)  # 500 نقطة لمنحنى ناعم
y_plot = regressor.predict(X_plot)                            # تنبؤات RF (متوسط الأشجار)

ax.scatter(X_train, y_train, color='blue', label='التدريب', s=80, zorder=3)   # نقاط التدريب
ax.scatter(X_test, y_test, color='green', label='الاختبار', s=80, zorder=3)       # نقاط الاختبار
ax.plot(X_plot, y_plot, color='red', linewidth=2, label='تنبؤ الغابة العشوائية')  # منحنى منعّم
ax.set_xlabel('Level')             # تسمية المحور السيني
ax.set_ylabel('Salary (USD)')     # تسمية المحور الصادي
ax.set_title('الغابة العشوائية — رواتب المناصب')  # عنوان الرسم
ax.legend()                        # عرض وسيلة الإيضاح

plt.tight_layout()  # ضبط المسافات
plt.show()          # عرض الرسم


## المثال 1 — الخلية 8: التقييم

تقييم الغابة العشوائية بـ MAE و RMSE و R² على مجموعة الاختبار.

**ما تفعله هذه الخلية:** حساب وعرض مقاييس التقييم.


In [ ]:
# الخطوة 8) التقييم
mae = mean_absolute_error(y_test, y_pred_test)              # متوسط الخطأ المطلق بالدولار
rmse = np.sqrt(mean_squared_error(y_test, y_pred_test))     # الجذر التربيعي لمتوسط مربع الخطأ
r2 = r2_score(y_test, y_pred_test)                          # التباين المُفسَّر

results = pd.DataFrame({
    'Metric': ['MAE', 'RMSE', 'R²'],
    'Value': [mae, rmse, r2],
    'Description': [
        'متوسط الخطأ المطلق (USD)',
        'الجذر التربيعي لمتوسط مربع الخطأ (USD)',
        'معامل التحديد (1 = مثالي)'
    ]
})

display(results.round(4))  # عرض جدول المقاييس
print(f'\nR² اختبار المثال 1 = {r2:.4f}')  # طباعة ملخص R²


## لماذا تعمل الغابة العشوائية لرواتب المناصب؟

| # | السبب | الشرح |
|---|--------|-------------|
| 1 | **قفزات راتب غير خطية** | عدة أشجار تلتقط أنماط خطوات مختلفة؛ المتوسط ينعّم التنبؤات |
| 2 | **تقليل الإفراط في التلائم** | التجميع يخفّض التباين مقارنة بـ CART عميقة واحدة على 10 صفوف |
| 3 | **لا حاجة للتحجيم** | الأشجار تقارن `Level ≤ threshold` — القيم الخام تعمل جيداً |
| 4 | **تنوع Bootstrap** | كل شجرة ترى عيّنة مختلفة قليلاً — مجموعة قوية |
| 5 | **تعميم أفضل** | عادةً تتفوق على شجرة واحدة على بيانات صغيرة وضوضائية |

> **الملخص:** الغابة العشوائية **تُوسط** عدة نماذج CART — أنعم وغالباً أدق من شجرة واحدة.


## فهم R² — المثال 1 (رواتب المناصب)

**R² (معامل التحديد)** يقيس مقدار تباين الراتب الذي يُفسَّر بواسطة Level.

| قيمة R² | المعنى |
|----------|---------|
| **1.0** | تنبؤات مثالية |
| **0.7–0.9** | ملاءمة قوية |
| **0.4–0.7** | ملاءمة متوسطة (شائعة مع مجموعات بيانات صغيرة جداً) |
| **0.0** | النموذج لا أفضل من التنبؤ بالمتوسط |
| **< 0** | أسوأ من المتوسط |

> مع **10 صفوف** فقط و**عينتين اختبار**، قد يتغيّر R² كثيراً — ركّز على **نمط** التنبؤات، وليس رقماً واحداً فقط.


---
# المثال 2: سعر المنزل — انحدار الغابة العشوائية

التنبؤ بـ **Price** من ثلاث ميزات للعقار باستخدام الغابة العشوائية مع **ميزات متعددة**.

| العمود | الدور | الوصف |
|--------|------|-------------|
| `Area_sqft` | ميزة (X₁) | مساحة المعيشة بالقدم المربع |
| `Bedrooms` | ميزة (X₂) | عدد غرف النوم |
| `Age_years` | ميزة (X₃) | عمر المنزل بالسنوات |
| `Price` | متغير مستهدف (y) | سعر البيع بالدولار |

**الملف:** `../Datasets/house_price.csv`


## المثال 2 — الخلية 1: تحميل واستكشاف البيانات

تحميل CSV أسعار المنازل وفحص البيانات.

**ما تفعله هذه الخلية:** قراءة `../Datasets/house_price.csv` وعرض إحصائيات أساسية.


In [ ]:
# الخطوة 1) تحميل مجموعة البيانات
dataset = pd.read_csv('../Datasets/house_price.csv')  # قراءة CSV إلى DataFrame

FEATURE_COLS = ['Area_sqft', 'Bedrooms', 'Age_years']  # ثلاث ميزات مدخلة
TARGET_COL = 'Price'                                    # المتغير المستهدف

print('أول 5 صفوف:')
display(dataset.head())

print('\nمعلومات مجموعة البيانات:')
dataset.info()

print('\nملخص إحصائي:')
display(dataset.describe())

print(f'\nالشكل: {dataset.shape[0]} صف × {dataset.shape[1]} عمود')


## المثال 2 — الخلية 2: تنظيف البيانات (معالجة القيم المفقودة)

تتضمن مجموعة البيانات هذه قيماً مفقودة لتوضيح `SimpleImputer`.

**ما تفعله هذه الخلية:** التحقق من القيم الفارغة، إزالة التكرارات، وإكمال القيم المفقودة.


In [ ]:
# الخطوة 2) تنظيف البيانات

print('القيم المفقودة لكل عمود:')
print(dataset.isnull().sum())

rows_before = len(dataset)
dataset = dataset.drop_duplicates().reset_index(drop=True)
rows_after = len(dataset)
print(f'\nالتكرارات المُزالة: {rows_before - rows_after}')

imputer = SimpleImputer(missing_values=np.nan, strategy='mean')
if dataset.isnull().sum().sum() > 0:
    dataset[FEATURE_COLS + [TARGET_COL]] = imputer.fit_transform(dataset[FEATURE_COLS + [TARGET_COL]])
    print('تم إكمال القيم المفقودة بالمتوسط')
else:
    print('لا توجد قيم مفقودة — لم يُطبَّق الإكمال')

print(f'\nالصفوف بعد التنظيف: {rows_after}')


## المثال 2 — الخلية 3: ترميز البيانات الفئوية

جميع الأعمدة عددية — يُتخطّى الترميز.

**ما تفعله هذه الخلية:** التأكد من عدم الحاجة للترميز الفئوي.


In [ ]:
# الخطوة 3) الترميز الفئوي

cat_cols = dataset.select_dtypes(include=['object', 'category']).columns.tolist()
if cat_cols:
    print(f'أعمدة فئوية وُجدت: {cat_cols}')
else:
    print('لا توجد أعمدة فئوية — تم تخطّي الترميز.')
    print(f'أعمدة الميزات: {FEATURE_COLS}')


## المثال 2 — الخلية 4: تقسيم البيانات

تعريف X (3 ميزات) و y (Price)، ثم تقسيم 80/20.

**ما تفعله هذه الخلية:** إنشاء مصفوفات الميزات/المستهدف وتطبيق train_test_split.


In [ ]:
# الخطوة 4) تقسيم التدريب-الاختبار

X = dataset[FEATURE_COLS].values  # مصفوفة الميزات: 3 أعمدة
y = dataset[TARGET_COL].values    # متجه المستهدف: قيم Price

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f'شكل X_train: {X_train.shape}')  # (n_train, 3)
print(f'شكل X_test:  {X_test.shape}')   # (n_test, 3)
print(f'شكل y_train: {y_train.shape}')
print(f'شكل y_test:  {y_test.shape}')


## المثال 2 — الخلية 5: تدريب النموذج

تدريب الغابة العشوائية مع ميزات متعددة — كل شجرة تقسم على مجموعات فرعية عشوائية من الميزات.

**ما تفعله هذه الخلية:** ملاءمة الغابة وعرض أهمية الميزات.


In [ ]:
# الخطوة 5) تدريب Random Forest Regressor

regressor = RandomForestRegressor(
    n_estimators=100,      # 100 شجرة في المجموعة
    max_depth=8,           # السماح بأشجار أعمق — بيانات أكثر من المثال 1
    min_samples_split=4,     # حاجة لـ 4 عيّنات على الأقل لتقسيم عقدة
    min_samples_leaf=2,      # كل ورقة يجب أن تحتوي على عيّنتين على الأقل
    max_features=1.0,        # استخدام جميع الميزات الثلاث (مجموعة ميزات صغيرة)
    random_state=42,
    n_jobs=-1                # تدريب متوازٍ عبر أنوية المعالج
)

regressor.fit(X_train, y_train)  # تدريب الغابة على جميع الميزات الثلاث

print('تم تدريب الغابة العشوائية بنجاح.')
print(f'عدد الأشجار: {len(regressor.estimators_)}')

print('\nأهمية الميزات:')  # متوسط عبر جميع الأشجار
for name, imp in zip(FEATURE_COLS, regressor.feature_importances_):
    print(f'  {name:15s} -> {imp:.4f}')  # أعلى = أكثر أهمية للتنبؤات


## المثال 2 — الخلية 6: التنبؤ

التنبؤ بأسعار المنازل على مجموعة الاختبار.

**ما تفعله هذه الخلية:** إنشاء تنبؤات المجموعة بمتوسط مخرجات الأشجار.


In [ ]:
# الخطوة 6) التنبؤ

y_pred_train = regressor.predict(X_train)
y_pred_test = regressor.predict(X_test)

print('عيّنة من التنبؤات (مجموعة الاختبار):')
for i in range(min(5, len(y_test))):
    print(f'  فعلي=${y_test[i]:,.0f}، متوقع=${y_pred_test[i]:,.0f}')


## المثال 2 — الخلية 7: التصوير

رسم **الفعلي مقابل المتوقع** و**أهمية الميزات**.

**ما تفعله هذه الخلية:** تصوير أداء النموذج وأي الميزات الأكثر أهمية.


In [ ]:
# الخطوة 7) التصوير

fig, axes = plt.subplots(1, 2, figsize=(14, 5))  # رسمتان فرعيتان

axes[0].scatter(y_test, y_pred_test, color='green', alpha=0.7)  # فعلي مقابل متوقع
min_val = min(y_test.min(), y_pred_test.min())
max_val = max(y_test.max(), y_pred_test.max())
axes[0].plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=2, label='تنبؤ مثالي')
axes[0].set_xlabel('السعر الفعلي (USD)')
axes[0].set_ylabel('السعر المتوقع (USD)')
axes[0].set_title('الفعلي مقابل المتوقع — سعر المنزل (الغابة العشوائية)')
axes[0].legend()

axes[1].barh(FEATURE_COLS, regressor.feature_importances_, color='darkgreen')  # رسم أهمية الميزات
axes[1].set_xlabel('الأهمية')
axes[1].set_title('أهمية الميزات (الغابة العشوائية)')

plt.tight_layout()
plt.show()


## المثال 2 — الخلية 8: التقييم

تقييم أداء الغابة العشوائية بـ MAE و RMSE و R².

**ما تفعله هذه الخلية:** حساب وعرض مقاييس التقييم.


In [ ]:
# الخطوة 8) التقييم
mae = mean_absolute_error(y_test, y_pred_test)
rmse = np.sqrt(mean_squared_error(y_test, y_pred_test))
r2 = r2_score(y_test, y_pred_test)

results = pd.DataFrame({
    'Metric': ['MAE', 'RMSE', 'R²'],
    'Value': [mae, rmse, r2],
    'Description': [
        'متوسط الخطأ المطلق (USD)',
        'الجذر التربيعي لمتوسط مربع الخطأ (USD)',
        'معامل التحديد (1 = مثالي)'
    ]
})

display(results.round(4))
print(f'\nR² اختبار المثال 2 = {r2:.4f}')


## لماذا تعمل الغابة العشوائية جيداً لسعر المنزل؟

| # | السبب | الشرح |
|---|--------|-------------|
| 1 | **ميزات متعددة** | الغابة تقسم على Area و Bedrooms و Age عبر عدة أشجار |
| 2 | **تفاعلات غير خطية** | الأشجار تلتقط مثلاً مساحة كبيرة + غرف نوم كثيرة → سعر أعلى |
| 3 | **أهمية الميزات** | تُظهر أي ميزة تساهم أكثر (متوسط عبر جميع الأشجار) |
| 4 | **لا حاجة للتحجيم** | القدم المربع وعدد الغرف والعمر الخام تعمل مباشرة |
| 5 | **R² قوي** | متوسط المجموعة عادةً يتفوق على CART واحدة على هذه المجموعة |

> **المقارنة:** الغابة العشوائية مقابل CART — نفس المقايضة في قابلية التفسير، لكن RF عادةً تعطي **دقة أعلى** و**إفراط تلائم أقل**.


## فهم R² — المثال 2 (سعر المنزل)

**R²** يخبرنا مدى جودة شرح Area و Bedrooms و Age معاً لأسعار المنازل.

| قيمة R² | المعنى |
|----------|---------|
| **> 0.85** | ممتاز — النموذج يلتقط معظم تباين السعر |
| **0.70–0.85** | ملاءمة جيدة لبيانات عقارية واقعية |
| **< 0.50** | ضعيف — فكّر في ميزات أكثر أو نموذج مختلف |

> الغابة العشوائية غالباً تحقق **R² أعلى** من شجرة قرار واحدة لأن المتوسط **يقلّل خطأ التنبؤ**.
